# 🔎 **Coleta de Dados**

## Origem dos Dados



A DEV Community é uma comunidade com mais de três milhões de desenvolvedores de software [<a href="#ref1">1</a>]. No site, os desenvolvedores escrevem artigos (posts de blog, perguntas em fóruns de discussão, tópicos de ajuda, etc.), participam de discussões e constroem seus perfis profissionais [<a href="#ref2">2</a>].

O site é hospedado pelo FOREM [<a href="#ref2">2</a>], um software de código aberto para a construção de comunidades. Na documentação do FOREM é disponibilizada uma API pública [<a href="#ref3">3</a>] que retorna os textos publicados, tags sobre o assunto tratado, entre outras informações. A documentação da API não estabelece uma licença específica para os dados disponibilizados. Os dados que podem identificar de alguma forma os autores dos textos foram ocultados.

Para realizar a coleta dos dados foi utilizada a versão 1 [<a href="#ref4">4</a>] da API pública do DEV Community. 

#### ⚖️ Governança de Dados, Ética e Licenciamento




Os dados brutos coletados por este script são obtidos de forma pública, respeitando as diretrizes de taxa de requisição da plataforma (*rate-limiting* via pausas controladas no código). 

Em conformidade com as políticas do site DEV.to, os artigos textuais consumidos estão sob a licença padrão **Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0)**.

#### 🔄 Consumo e Retorno da API



⚠️ Descrição pendente
* incluir sobre os parametros de consulta da API 
* incluir sobre os dados de retorno> da API 

## O Problema

Para realizar o projeto TechMind no Hackathon ONE, é necessário obter textos técnicos da área de tecnologia para treinar um modelo de classificação de textos. O modelo irá classificar os textos pelo assunto principal, que aqui estamos chamando de categoria. Os dados obtidos, através da API, não possuem um target que represente a categoria. Porém os artigos publicados no site Dev Community, possuem tags associadas a eles, que representam o tema do artigo. Conforme a documentação da API, essas tags podem ser utilizadas nos parametros da API para obter artigos especificos. Dito isso, se faz necessário realizar a Rotulagem de Dados (Data Labeling), gerando assim o target de cada texto. 

Além disso, será necessária a extração de uma amostra dos dados para validação manual dos textos, afim de conferir se os mesmos são realmente da categoria a que foi associado. Dessa forma iremos validar a qualidade dos dados e consequentemente viabilizar a utilização dos mesmos no projeto. 

## Objetivo



Este notebook tem como objetivo:

* realizar a coleta automatizada de artigos técnicos consumindo o endpoint público /api/articles da versão 1 [<a href="#ref4">4</a>]da API, usando como parâmetro as tags associadas a eles.
* filtrar somente os artigos em inglês.
* realizar a rotulação dos dados, associando um target a cada texto obtido.
* obter uma amostra dos dados para posterior validação manual da qualidade dos textos.

A estratégia foi desenhada para capturar dados intencionalmente desbalanceados e com ruídos bi-idioma, simulando um cenário real de engenharia de dados.

## Coleta e Data Labeling

#### Quantidade de Dados

Para realizar um MVP iremos coletar cerca de 1800 registros, sendo 300 de cada categoria. 
Considerando que a maior parte dos materiais de temas que envolvem tecnologia serem disponibilizados na lingua inglesa, o modelo será treinado nessa lingua. 

#### Rotulagem

Para realizar a rotulagem dos textos, foram definidas 6 categorias e, para cada uma delas, 5 subcategorias: 

⚠️ Descrição pendente
* incluir descrição das catergoris e subcategorias>


Cada subcategoria será utilizada como parametro (tag) no consumo da API. Os textos retornados serão associados a categoria que representa a subcategoria, gerando assim o target.



In [ ]:
# Importando bibliotecas
import requests
import time
import pandas as pd
from langdetect import detect
import json
import re
import os

from IPython.display import display
pd.reset_option('display.max_colwidth')

In [ ]:
# Estrutura Hierárquica: 6 Categorias Principais, cada uma com 5 Subtags
mapeamento_categorias = {
    'Frontend': ['javascript', 'react', 'css', 'typescript', 'nextjs'],
    'Backend': ['springboot', 'java', 'nodejs', 'python', 'csharp'],
    'Data Science': ['datascience', 'machinelearning', 'pandas', 'ai', 'dataanalysis'],
    'Cloud': ['oci', 'aws', 'azure', 'devops', 'docker'],
    'Database': ['postgres', 'mysql', 'sql', 'mongodb', 'oracle'],
    'Security': ['cybersecurity', 'security', 'infosec', 'cryptography', 'ethicalhacking']
}

In [ ]:
# Buscar artigos por categoria
artigos_dataset = []
total_tentativas = 0

print("🚀 Iniciando extração de Conteúdo Completo da API DEV.to...")
print("-------------------------------------------------------")

for categoria, subtags in mapeamento_categorias.items():
    print(f"\n📁 Coletando para a Categoria Principal: [{categoria}]")
    
    for tag in subtags:
        print(f"  └─ 🏷️ Buscando subtag: {tag}")
        
        # Páginas 1 e 2 trazendo 30 itens em cada uma para mirar nos 60 por subtag
        for pagina in [1, 2]:
            url_lista = f"https://dev.to/api/articles?tag={tag}&per_page=30&page={pagina}"
            
            try:
                resposta_lista = requests.get(url_lista).json()
                
                if not resposta_lista or not isinstance(resposta_lista, list):
                    break
                    
                for artigo in resposta_lista:
                    total_tentativas += 1
                    artigo_id = artigo['id']
                    
                    # --- PASSO ADICIONAL: Requisitar o conteúdo completo do artigo ---
                    url_artigo_completo = f"https://dev.to/api/articles/{artigo_id}"
                    detalhe = requests.get(url_artigo_completo).json()
                    
                    # O conteúdo completo vem no campo 'body_markdown'
                    corpo_texto = detalhe.get("body_markdown", "")
                    
                    if not corpo_texto:
                        continue
                        
                    # Filtro de Idioma aplicado sobre o texto completo do artigo
                    try:
                        idioma = detect(corpo_texto[:1000]) # Detecta usando os primeiros 1000 caracteres para ser mais rápido
                    except:
                        idioma = "unknown"
                        
                    # Bloco com a sua estrutura exata de chaves e variáveis ajustadas
                    if idioma == 'en':
                        artigos_dataset.append({
                            "categoria": categoria,
                            "titulo": detalhe.get("title", ""),
                            #"descricao": artigo.get("description", ""),
                            "texto": detalhe.get("body_markdown", ""), 
                            "tags": detalhe.get("tag_list", []),
                            "fonte": artigo.get("url", "")
                        })
                        
                    # Pausa micro entre requisições de detalhes para não estressar a API
                    time.sleep(0.2)
                        
            except Exception as e:
                print(f"     ❌ Erro na tag {tag} (Pág {pagina}): {e}")
                
        # Pausa regulamentar entre subtags
        time.sleep(0.5)

🚀 Iniciando extração de Conteúdo Completo da API DEV.to...
-------------------------------------------------------

📁 Coletando para a Categoria Principal: [Frontend]
  └─ 🏷️ Buscando subtag: javascript
  └─ 🏷️ Buscando subtag: react
  └─ 🏷️ Buscando subtag: css
  └─ 🏷️ Buscando subtag: typescript
  └─ 🏷️ Buscando subtag: nextjs

📁 Coletando para a Categoria Principal: [Backend]
  └─ 🏷️ Buscando subtag: springboot
  └─ 🏷️ Buscando subtag: java
  └─ 🏷️ Buscando subtag: nodejs
  └─ 🏷️ Buscando subtag: python
  └─ 🏷️ Buscando subtag: csharp

📁 Coletando para a Categoria Principal: [Data Science]
  └─ 🏷️ Buscando subtag: datascience
  └─ 🏷️ Buscando subtag: machinelearning
  └─ 🏷️ Buscando subtag: pandas
  └─ 🏷️ Buscando subtag: ai
  └─ 🏷️ Buscando subtag: dataanalysis

📁 Coletando para a Categoria Principal: [Cloud]
  └─ 🏷️ Buscando subtag: oci
  └─ 🏷️ Buscando subtag: aws
  └─ 🏷️ Buscando subtag: azure
  └─ 🏷️ Buscando subtag: devops
  └─ 🏷️ Buscando subtag: docker

📁 Coletando para a Ca

In [4]:
# 2. Construção do DataFrame Bruto
df = pd.DataFrame(artigos_dataset)

In [5]:
# 3. Salvando o arquivo bruto
df.to_csv('../dados/dataset_devto_v1.csv', index=False)

print("\n-------------------------------------------------------")
print("🏆 EXTRAÇÃO DE CONTEÚDO COMPLETO CONCLUÍDA!")
print(f"Total de artigos processados pela API: {total_tentativas}")
print(f"Total de artigos salvos (em Inglês): {len(df)}")


-------------------------------------------------------
🏆 EXTRAÇÃO DE CONTEÚDO COMPLETO CONCLUÍDA!
Total de artigos processados pela API: 1785
Total de artigos salvos (em Inglês): 1626


In [6]:
print("\n---------------------------------------")
print("\n📊 Distribuição Bruta por Categoria:")
print(df['categoria'].value_counts())


---------------------------------------

📊 Distribuição Bruta por Categoria:
categoria
Security        288
Frontend        287
Cloud           280
Database        268
Data Science    266
Backend         237
Name: count, dtype: int64


## Análise Exploratória

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1626 entries, 0 to 1625
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   categoria  1626 non-null   str  
 1   titulo     1626 non-null   str  
 2   texto      1626 non-null   str  
 3   tags       1626 non-null   str  
 4   fonte      1626 non-null   str  
dtypes: str(5)
memory usage: 11.6 MB


In [8]:
df.head()

,categoria,titulo,texto,tags,fonte
0,Frontend,🚀 Day 7 of Learning React: How Do Components S...,📌 *Missed Day 6? I explored how React renders ...,"react, javascript, webdev, beginners",https://dev.to/bismay-exe/day-7-of-learning-re...
1,Frontend,"There are around 10 independent PDF engines, a...",I'm a student who mostly built little apps fo...,"webdev, programming, productivity, javascript",https://dev.to/keypdf_official/there-are-aroun...
2,Frontend,How We Built DJ ROOTS: An AI-Powered Music Rec...,# 🎧 DJ ROOTS – Building a Real-Time Collaborat...,"webdev, javascript, react, ai",https://dev.to/ananyasharma_14/how-we-built-dj...
3,Frontend,JavaScript Atomics and SharedArrayBuffer in 20...,# JavaScript Atomics and SharedArrayBuffer in ...,"javascript, webdev, react, programming",https://dev.to/jsmanifest/javascript-atomics-a...
4,Frontend,Buiding Browser-Based SBOM Visualizer,Software Bills of Materials are becoming incre...,"devops, docker, software, javascript",https://dev.to/greedykomododragon/buiding-brow...


## Amostragem Estratificada dos Dados

In [9]:
print("🎯 Iniciando amostragem estratificada com conversão robusta...")

# --- DIAGNÓSTICO E LIMPEZA DE FORMATO ---
# Remove aspas extras que o CSV coloca nas pontas da lista (ex: '"['tag']"' vira '[tag]')
def extrair_lista_tags(valor):
    if isinstance(valor, list):
        return valor
    if pd.isnull(valor):
        return []
    
    texto = str(valor).strip()
    
    # Se o texto já se parece com uma lista mapeada por strings
    try:
        # Se contiver aspas simples, padroniza para aspas duplas para o json entender
        texto_json = texto.replace("'", '"')
        return json.loads(texto_json)
    except:
        # Se falhar o JSON, limpa colchetes e aspas usando Regex e quebra por vírgula
        tags_limpas = re.sub(r"[\[\]'\" ]", "", texto)
        return tags_limpas.split(",") if tags_limpas else []

# Aplica a limpeza na coluna 'tags'
df['tags'] = df['tags'].apply(extrair_lista_tags)

# Mostra uma amostra rápida do resultado da conversão para termos certeza absoluta
print(f"🔬 Tipo atual do dado na primeira linha: {type(df['tags'].iloc[0])}")
print(f"📝 Exemplo de conteúdo convertido: {df['tags'].iloc[0]}")
print("---------------------------------------------------------------------------------")

quantidade_por_subtag = 2
lista_amostra = []

# Varre a estrutura original de categorias para fazer o filtro
for categoria, subtags in mapeamento_categorias.items():
    for tag_alvo in subtags:
        
        # Filtro tolerante a maiúsculas/minúsculas e espaços em branco
        df_filtrado = df[
            (df['categoria'] == categoria) & 
            (df['tags'].apply(lambda lista: tag_alvo.lower().strip() in [str(t).lower().strip() for t in lista] if isinstance(lista, list) else False))
        ]
        
        # Se encontrou, adiciona à amostra
        if not df_filtrado.empty:
            tamanho_real_amostra = min(quantidade_por_subtag, len(df_filtrado))
            artigo_sorteado = df_filtrado.sample(n=tamanho_real_amostra, random_state=42).copy()
            artigo_sorteado['tag_validada'] = tag_alvo
            lista_amostra.append(artigo_sorteado)

# Consolida se houver dados
if len(lista_amostra) > 0:
    df_amostra_validacao = pd.concat(lista_amostra, ignore_index=True)
    # df_amostra_validacao['resumo_texto'] = df_amostra_validacao['texto'].str.slice(0, 120) + "..."
    df_amostra_validacao[['categoria', 'tag_validada', 'titulo', 'texto', 'tags', 'fonte']].to_csv('../dados/amostra_dataset_devto_v1.csv', index=False)

    print(f"✅ Sucesso! Arquivo 'amostra_validacao_artigos.csv' gerado com {len(df_amostra_validacao)} artigos.")
    display(df_amostra_validacao[['categoria', 'tag_validada', 'titulo', 'texto', 'tags', 'fonte']])
else:
    print("❌ O filtro ainda não encontrou correspondências.")
    print("Vamos dar uma olhada nas primeiras linhas da coluna para entender o formato:")
    print(df[['categoria', 'tags']].head(5))

🎯 Iniciando amostragem estratificada com conversão robusta...
🔬 Tipo atual do dado na primeira linha: <class 'list'>
📝 Exemplo de conteúdo convertido: ['react', 'javascript', 'webdev', 'beginners']
---------------------------------------------------------------------------------
✅ Sucesso! Arquivo 'amostra_validacao_artigos.csv' gerado com 60 artigos.


,categoria,tag_validada,titulo,texto,tags,fonte
0,Frontend,javascript,What broke in the modern web stack this week (...,517 changes shipped across 30 tracked develope...,"[webdev, javascript, changelog]",https://dev.to/stacktraceweekly/what-broke-in-...
1,Frontend,javascript,Component Deep Dive #40: Pagination — The Most...,**Web Component Dictionary v2.0** 收录83个手写UI组件—...,"[component, pagination, css, javascript]",https://dev.to/wdsega/component-deep-dive-40-p...
2,Frontend,react,FutboLeyendas - El Olimpo del Fútbol con Gemin...,# FootLegends⚽🤖 \n\n## Our Passion Project\nFo...,"[weekendchallenge, googleaichallenge, react, j...",https://dev.to/eliezer_josepolidorsola/futbole...
3,Frontend,react,The ForgeStack Release Wave: Four New Librarie...,**Source and repos:** [github.com/yaghobieh](h...,"[react, ai, node, webdev]",https://dev.to/john_yaghobieh_8f294091f6/the-f...
4,Frontend,css,"Introducing Popkorn: a portable, CSS-based for...",\nCSS is capable of expressing a lot when it c...,"[css, lottie, animation, javascript]",https://dev.to/ayarse/introducing-popkorn-a-po...
5,Frontend,css,Building a CSS Structure That Stays Maintainable,## How to Organize CSS Like a Professional\n\n...,"[css, webdev, programming, tutorial]",https://dev.to/jsdevspace/building-a-css-struc...
6,Frontend,typescript,Stripe to Mollie Migration: What Actually Breaks,I did not switch to Mollie because I love it. ...,"[nextjs, typescript, node, drizzle]",https://dev.to/iurii_rogulia/stripe-to-mollie-...
7,Frontend,typescript,Type-safe Elasticsearch queries in TypeScript ...,"---\ntitle: ""Type-safe Elasticsearch queries i...","[typescript, elasticsearch, javascript, builder]",https://dev.to/john_rodger_dee953ed28186/type-...
8,Frontend,nextjs,Unblock the Main Thread: Partytown in Next.js ⚡,<h2>The Third-Party Performance Trap</h2>\n<p>...,"[nextjs, react, webperf, frontend]",https://dev.to/iprajapatiparesh/unblock-the-ma...
9,Frontend,nextjs,How I Built My Next.js Developer Portfolio & O...,When most developers build their personal webs...,"[nextjs, seo, webdev, javascript]",https://dev.to/ruumidev/how-i-built-my-nextjs-...


## Primeiros Insights

A execução bem-sucedida dos scripts de coleta e amostragem resultou na estruturação e armazenamento dos dados em dois arquivos CSV distintos, garantindo a organização do projeto:

 1. `dataset_devto_v1.csv` (Base Completa): Contém os 1.634 artigos filtrados em inglês pela API. Este arquivo preserva os dados originais e será a base oficial de treinamento do modelo.

 2. `amostra_dataset_devto_v1.csv` (Base de Auditoria): Um arquivo menor contendo exatamente 60 artigos sorteados aleatoriamente (2 por subtag). Ele foi gerado especificamente para permitir o controle de qualidade visual do projeto.

A partir da análise dessas duas bases e da inspeção visual humana realizada na amostra de 60 artigos (com auxílio de ferramentas de tradução), foram gerados os seguintes insights:

* **Aderência dos Temas:** A checagem humana confirmou que o conteúdo dos artigos é altamente qualificado e bate exatamente com as categorias pretendidas.

* **Múltiplas Tags:** Os autores utilizam várias tags por post (ex: um artigo de `#react` também traz `#frontend` e `#javascript`), provando que os dados capturaram o ecossistema real e interligado da tecnologia.

* **Presença de Elementos de Programação:** Os textos extraídos vêm acompanhados de blocos de códigos reais de programação (sintaxes de software), links e marcações visuais de Markdown.

## Conclusão e Recomendações

Com base nos insights obtidos na fase de obtenção de dados, conclui-se que o banco de dados é estatisticamente rico e viável, mas exige cuidados antes de alimentar a inteligência artificial.

Como o modelo será treinado apenas com textos explicativos (vocabulário humano), recomenda-se as seguintes ações para a próxima etapa do projeto:

* **Remoção de Códigos de Programação:** Criar um script de limpeza para deletar os exemplos de códigos colados pelos autores, evitando que termos como const, return ou chaves {} confundam o classificador.

* **Limpeza de Formatação:** Aplicar filtros para apagar links, imagens e caracteres especiais do Markdown que atuam como ruído.

* **Normalização Textual:** Converter todas as palavras para minúsculas e aplicar técnicas de Processamento de Linguagem Natural (PLN) para preparar o texto final.

## Referências



* <a id="ref1"></a>**[1]** DEV COMMUNITY. *Site*. Disponível em: <https://dev.to/>. Acesso em: 9 jul. 2026.

* <a id="ref2"></a>**[2]** FOREM. *Sobre dev.to*. Disponível em: <https://github.com/forem/forem/>. Acesso em: 9 jul. 2026.

* <a id="ref3"></a>**[3]** API. *Versões da API*. Disponível em: <https://developers.forem.com/api>. Acesso em: 9 jul. 2026.

* <a id="ref4"></a>**[4]** Forem API V1. *Documentação da API*. Disponível em: <https://developers.forem.com/api/v1>. Acesso em: 9 jul. 2026.